# Local the model
Download it from gradio and move it to the GPU

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
DEVICE = "mps"

print(f"Loading {MODEL_ID} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
).to(DEVICE)
model.eval()
print("Loaded.")

Loading deepseek-ai/DeepSeek-R1-Distill-Qwen-14B on mps...


Loading weights: 100%|██████████| 579/579 [00:00<00:00, 7833.52it/s]


Loaded.


# generate() test

In [ ]:
messages = [{"role": "user", "content": "What's 2+2? Emit some tokens in your thinking block too."}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
out = model.generate(**inputs, max_new_tokens=500, do_sample=True, temperature=0.6, top_p=0.95)
print(tokenizer.decode(out[0], skip_special_tokens=False))

In [6]:
print(f"model: {model}")
with torch.inference_mode():
   outputs = model(**inputs, output_hidden_states=True)

   print(type(outputs))
   print("logits shape:", outputs.logits.shape)
   print("num hidden states:", len(outputs.hidden_states))
   print("each hidden state shape:", outputs.hidden_states[0].shape)

model: Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 5120)
    (layers): ModuleList(
      (0-47): 48 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=True)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=True)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (up_proj): Linear(in_features=5120, out_features=13824, bias=False)
          (down_proj): Linear(in_features=13824, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm): Qwen2RMSNorm((5120,), eps=1e-05)
      )
    )
    (norm): Qwen2RMSNorm((5120,), eps=1e-05